In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage
from langchain_core.tools import tool, Tool
from pydantic import BaseModel
from langchain.mcp import MCPAdapter



import subprocess
import sys 
from uuid import uuid4
import json
import asyncio

### Build tool registry JSON from function definitions.
The LLM can search the registry, fetch and execute scripts, and push new scripts to the registry


In [6]:
from src.tool_registry import ScriptRegistry, build_registry_tools

registry = ScriptRegistry()
tools = build_registry_tools(registry)

# After reviewing a generated script, enable it explicitly:
# registry.enable_script("the-id-returned-by-write_script_tool")


### Add RAG tools to allow the model to search documents.

In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings

In [ ]:
import requests
def load():
    paths = [
        'https://docs.langchain.com/oss/python/deepagents/rag',
    ]
    docs : list[Document] = []
    for path in paths:
        try:
            response = requests.get(path,timeout=15)
            response.raise_for_status()
        except requests.RequestException as e:
            continue
        source = path
        docs.append(
            Document(page_content=response.text, metadata={'source':source})
        )

    return docs

def split(docs: list[Document]):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
    splits = text_splitter.split_documents(docs)
    return splits





In [35]:
import os 

docs = load()
splits = split(docs)
embedding = OllamaEmbeddings(model='qwen3-embedding:0.6b',)
vector_store = InMemoryVectorStore(embedding)
vector_store.add_documents(documents=splits)

[]

In [31]:
from uuid import uuid4
import chromadb 

docs = [
    {'id':'doc1', 'text':'hello, world', 'metadata':{'source':'tweet'}},
    {'id':'doc2', 'text':'the square root of nine is three', 'metadata':{'source':'mathhub'}},
    {'id':'doc3', 'text':'The number 42 is the universe number, I guess', 'metadata':{'source':'tweet'}},
]


SAMPLE_DOCS = [
    Document(page_content=x['text'], metadata=x['metadata']) for x in docs
]
client = chromadb.Client()

collection = client.get_or_create_collection('test_collection')

for doc in docs:
    collection.upsert(ids=doc['id'], documents=[doc['text']])



In [ ]:
import tempfile
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import ChatOllama

embedding_model = OllamaEmbeddings(model='qwen3-embedding:0.6b')
def similarity_search_with_scores():
    with tempfile.TemporaryDirectory() as dir:
        print(dir)
        vectorstore = Chroma.from_documents(
            documents= SAMPLE_DOCS,
            embedding=embedding_model,
            persist_directory=dir,
        )

        query = 'Hello, World'
        retr = vectorstore.as_retriever(search_type='similarity',kwargs={'k':2})
        results = retr.invoke(query, k=3,filter={'source':'tweet'})
        for i, (doc, score) in enumerate(results):
            print(f"Document {i} : content: {doc.page_content}, score={score}")

similarity_search_with_scores()

/tmp/tmpawt_f_0y
Document 0 : content: hello, world, score=0.06739351153373718
Document 1 : content: The number 42 is the universe number, I guess, score=1.060866355895996


In [99]:
import requests
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough # like idenetity function def f(x):return x
def create_knowledge_base() -> Chroma:
 
    docs = SAMPLE_DOCS

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(docs)

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=tempfile.mkdtemp()
    )
    return vectorstore

def rag_pipeline():
    vectorstore = create_knowledge_base()
    retriever = vectorstore.as_retriever(search_type='similarity', kwargs={"k":2})
    llm = ChatOllama(model='qwen3.5:4b',
                     temperature=0.2,
                     )
    prompt = ChatPromptTemplate.from_template(
        """Answer the question based on the following context:
        
        {context}
        
        Question: {question}
        Answer:
        
        Only used the available context. If you don't know the answer, just respond with "I don't know". Be concise
        """
    )

    def format_docs(docs):
        return "\n\n".join([doc.page_content for doc in docs])

    rag_chain = (
        {'context': retriever | format_docs, 'question': RunnablePassthrough}
        | prompt
        | llm 
        | StrOutputParser()
    )
    ans = retriever.invoke('hello, world')
    print(format_docs(ans))
    answer = rag_chain.invoke("Hello, World")
    print(answer)

In [98]:
rag_pipeline()

hello, world

The number 42 is the universe number, I guess

the square root of nine is three


ValidationError: 2 validation errors for RunnablePassthrough
func.callable
  Input should be callable [type=callable_type, input_value='Hello, World', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/callable_type
func.callable
  Input should be callable [type=callable_type, input_value='Hello, World', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/callable_type

In [41]:
prompt = ChatPromptTemplate.from_template(
        """Answer the question based on the following context:
        
        {context}
        
        Question: {question}
        Answer:
        
        Only used the available context. If you don't know the answer, just respond with "I don't know". Be concise
        """
    )
prompt.invoke(input={'question':'hello', 'context':'Nothing'})

ChatPromptValue(messages=[HumanMessage(content='Answer the question based on the following context:\n\n        Nothing\n\n        Question: hello\n        Answer:\n\n        Only used the available context. If you don\'t know the answer, just respond with "I don\'t know". Be concise\n        ', additional_kwargs={}, response_metadata={})])

ValueError: Expected document to be a str, got page_content='Hello, Mr Robinson' metadata={'source': 'tweet'} in add.

In [18]:
chat = ChatOllama(
    model="qwen3.5:4b",
    temperature=0,
    max_tokens=2048,
)

SystemPrompt = """You are a voice assistant with a persistent Python tool registry.
Search with search_scripts before creating a tool. Use execute_script with an
existing enabled tool's ID and schema-compatible arguments when it fits.
If no suitable tool exists, use write_script_tool to save a new script. Give
its entry-point function typed named parameters, a return annotation, a clear
docstring, and a JSON-serializable result. script_name has no path or extension.
New scripts are pending_validation: explain that review and enabling are needed
before execution. Never claim a pending script has run. Tool failures are
returned as error objects; read them and correct the request when appropriate.
"""
tools = build_registry_tools(registry)

messages = [HumanMessage(content="Print the word Hell after 3 seconds.")]
messages_write_python_script = [HumanMessage(
    content="list the available tools and use the appropriate one to solve the expressoin 43*(123+43/3)^2 step by step"
)]


async def llm_chat(messages: list, tools: list | None = None, max_rounds: int = 8) -> list:
    
    mcp_config = {
            "mcpServers": {
                "weather": {
                    "command": "npx",
                    "args": ["-y", "@dangahagan/weather-mcp"],
                    "env": {"WEATHER_UNITS": "metric"},
                },
                "sequential-thinking": {
                      "command": "npx",
                      "args": [
                        "-y",
                        "@modelcontextprotocol/server-sequential-thinking"
                      ]
                },
            }
        }
    async with MCPAdapter(mcp_config) as adapter:
        mcp_tools = await adapter.list_tools()
        tools.extend(mcp_tools)
        available = {registered_tool.name: registered_tool for registered_tool in (tools or [])}
        model_with_tools = chat.bind_tools(list(available.values()))
        for _ in range(max_rounds):
            result = model_with_tools.invoke([SystemMessage(content=SystemPrompt)] + messages)
            messages.append(result)
            if result.content:
                print(f"[AGENT]: {result.content}")
            if not result.tool_calls:
                return messages
            for call in result.tool_calls:
                print(f"[TOOL CALL]: {call['name']}")
                try:
                    registered_tool = available[call["name"]]
                    tool_result = await registered_tool.ainvoke(call["args"])
                except Exception as exc:
                    tool_result = {"error": str(exc)}
                print(f"[OBS]: {tool_result}")
                messages.append(ToolMessage(
                    content=json.dumps(tool_result), tool_call_id=call["id"]
                ))
        raise RuntimeError("Agent reached its tool-call round limit.")


In [19]:
result = await llm_chat(messages_write_python_script, tools)

[10/25/26 11:36:11] INFO     Proxy detected connected client -     ]8;id=1231061;file:///home/bashar/Study/Projects/AIVoiceAssistant/venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=1231062;file:///home/bashar/Study/Projects/AIVoiceAssistant/venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py#1371\1371]8;;\
                             reusing existing session for all                   
                             requests. This may cause context                   
                             mixing in concurrent scenarios, and                
                             the session's existing settings                    
                             apply, so backend results are                      
                             validated against their declared                   
                             output schema rather than relayed                  
                             as-is. Pass a disconnected clien

[AGENT]: Looking at my available tools, I can see:

**Available Tools:**
- `weather_get_alerts` - Get active weather alerts for locations
- `weather_search_location` - Search for locations by name to get coordinates
- `weather_check_service_status` - Check upstream weather API status
- `sequential-thinking_sequentialthinking` - A detailed tool for dynamic and reflective problem-solving through thoughts

None of these tools are designed to solve mathematical expressions. However, I can use the sequential thinking tool to work through this expression step by step.

Let me solve **43*(123+43/3)^2** step by step:
[TOOL CALL]: sequential-thinking_sequentialthinking
[OBS]: [{'type': 'text', 'text': '{\n  "thoughtNumber": 1,\n  "totalThoughts": 5,\n  "nextThoughtNeeded": true,\n  "branches": [],\n  "thoughtHistoryLength": 1\n}', 'id': 'lc_53bc5f9b-5190-4ea9-b675-85c92e4679c2'}]


HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[AGENT]: I see you've started a sequential thinking process. To continue effectively, I need to understand what specific problem or task we're working on. 

Could you clarify:
- What weather-related question or task do you need help with?
- Are you looking for alerts, forecasts, location information, or something else?

Once I know the goal, I can continue the thinking process and use the appropriate tools to get you the information you need.


#### OKAY, we have a primitive (but working) llm agent, now let's add the ASR model:

In [20]:
from qwen_asr import Qwen3ASRModel
import torch

model = Qwen3ASRModel.from_pretrained(
    "Qwen/Qwen3-ASR-0.6B",
    dtype=torch.bfloat16,
    device_map="cuda:0",
    # attn_implementation="flash_attention_2",
    max_inference_batch_size=32, # Batch size limit for inference. -1 means unlimited. Smaller values can help avoid OOM.
    max_new_tokens=256, # Maximum number of tokens to generate. Set a larger value for long audio input.
)



In [21]:
## Record 5-second audio
import sounddevice as sd
from scipy.io.wavfile import write

sample_rate = 24000
duration = 5

def record_audio(sr, duration) :
    print(f"Recording audio for {duration} seconds...")
    audio = (sd.rec(
        int(duration * sr),
        samplerate=sr,
        channels=1,
        dtype="int16",
    ), sample_rate)
    sd.wait()
    return audio

In [22]:
%time
results = model.transcribe(
    audio=audio,
    language="English"
)
print(results)

CPU times: user 10 μs, sys: 3 μs, total: 13 μs
Wall time: 18.6 μs


NameError: name 'audio' is not defined

### Combine audio -> ASR -> LLM -> Output

In [31]:
audio = record_audio(sr=24000, duration=10)
text = model.transcribe(audio=audio, language="English")[0].text
messages = [HumanMessage(content= text)]
llm_result = await llm_chat(messages=messages, tools=tools)


Recording audio for 10 seconds...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
[10/25/26 11:39:29] INFO     Proxy detected connected client -     ]8;id=1231085;file:///home/bashar/Study/Projects/AIVoiceAssistant/venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=1231086;file:///home/bashar/Study/Projects/AIVoiceAssistant/venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py#1371\1371]8;;\
                             reusing existing session for all                   
                             requests. This may cause context                   
                             mixing in concurrent scenarios, and                
                             the session's existing settings                    
                             apply, so backend results are                      
                             validated against their declared                   
                             output schema rather than relayed       

[AGENT]: I notice your request appears incomplete - you mentioned "returns the." but didn't specify what the script should do with the list of strings or what it should return.

Could you please clarify:
1. What operation should be performed on the list of strings? (e.g., join, filter, transform, etc.)
2. What should the script return? (e.g., a single string, a modified list, counts, etc.)

Once you provide these details, I'll create a properly typed script for you.


In [32]:
llm_result

[HumanMessage(content='Now create a script that takes as an argument a list of strings and returns the.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='I notice your request appears incomplete - you mentioned "returns the." but didn\'t specify what the script should do with the list of strings or what it should return.\n\nCould you please clarify:\n1. What operation should be performed on the list of strings? (e.g., join, filter, transform, etc.)\n2. What should the script return? (e.g., a single string, a modified list, counts, etc.)\n\nOnce you provide these details, I\'ll create a properly typed script for you.', additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-10-25T08:39:35.437078016Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5085081704, 'load_duration': 1382925, 'prompt_eval_count': 2050, 'prompt_eval_duration': 640532000, 'eval_count': 356, 'eval_duration': 4301743000, 'logprobs': None, 'model_name': 'qwen3.5:4

### The Pipeline looks fairly good, we will now deal with voice verification 

In [35]:
import torchaudio
from speechbrain.inference.speaker import EncoderClassifier

classifier = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb")
signal, fs =torchaudio.load('audio_wavs/recording.wav')
print(signal.shape)
signal = signal.mean(dim=0, keepdim=True)
signal = torchaudio.functional.resample(signal, fs, 16000) # necessary for ecapa-tdnn
embeddings = classifier.encode_batch(signal)


Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


torch.Size([1, 120000])


In [53]:
classifier

EncoderClassifier(
  (mods): ModuleDict(
    (compute_features): Fbank(
      (compute_STFT): STFT()
      (compute_fbanks): Filterbank()
      (compute_deltas): Deltas()
      (context_window): ContextWindow()
    )
    (mean_var_norm): InputNormalization()
    (embedding_model): ECAPA_TDNN(
      (blocks): ModuleList(
        (0): TDNNBlock(
          (conv): Conv1d(
            (conv): Conv1d(80, 1024, kernel_size=(5,), stride=(1,))
          )
          (activation): ReLU()
          (norm): BatchNorm1d(
            (norm): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          )
          (dropout): Dropout1d(p=0.0, inplace=False)
        )
        (1): SERes2NetBlock(
          (tdnn1): TDNNBlock(
            (conv): Conv1d(
              (conv): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
            )
            (activation): ReLU()
            (norm): BatchNorm1d(
              (norm): BatchNorm1d(1024, eps=1e-05, momentum

In [ ]:
from speechbrain.inference.speaker import SpeakerRecognition
verification = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="pretrained_models/spkrec-ecapa-voxceleb")
score, prediction = verification.verify_files("audio_wavs/recording.wav", "audio_wavs/recording_dif.wav") # Different Speakers
print(f"{score= }, {prediction= }")
score, prediction = verification.verify_files("audio_wavs/recording.wav", "audio_wavs/recording_sim.wav") # Same Speaker
print(f"{score= }, {prediction= }")

Fetch hyperparams.yaml: Using symlink found at '/home/bashar/Study/Projects/AIVoiceAssistant/pretrained_models/spkrec-ecapa-voxceleb/hyperparams.yaml'
Fetch embedding_model.ckpt: Using symlink found at '/home/bashar/Study/Projects/AIVoiceAssistant/pretrained_models/spkrec-ecapa-voxceleb/embedding_model.ckpt'
Fetch mean_var_norm_emb.ckpt: Using symlink found at '/home/bashar/Study/Projects/AIVoiceAssistant/pretrained_models/spkrec-ecapa-voxceleb/mean_var_norm_emb.ckpt'
Fetch classifier.ckpt: Using symlink found at '/home/bashar/Study/Projects/AIVoiceAssistant/pretrained_models/spkrec-ecapa-voxceleb/classifier.ckpt'
Fetch label_encoder.txt: Using symlink found at '/home/bashar/Study/Projects/AIVoiceAssistant/pretrained_models/spkrec-ecapa-voxceleb/label_encoder.ckpt'
Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


score= tensor([0.2358], device='cuda:0'), prediction= tensor([False], device='cuda:0')
score= tensor([0.7795], device='cuda:0'), prediction= tensor([True], device='cuda:0')


In [52]:
verification

SpeakerRecognition(
  (mods): ModuleDict(
    (compute_features): Fbank(
      (compute_STFT): STFT()
      (compute_fbanks): Filterbank()
      (compute_deltas): Deltas()
      (context_window): ContextWindow()
    )
    (mean_var_norm): InputNormalization()
    (embedding_model): ECAPA_TDNN(
      (blocks): ModuleList(
        (0): TDNNBlock(
          (conv): Conv1d(
            (conv): Conv1d(80, 1024, kernel_size=(5,), stride=(1,))
          )
          (activation): ReLU()
          (norm): BatchNorm1d(
            (norm): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          )
          (dropout): Dropout1d(p=0.0, inplace=False)
        )
        (1): SERes2NetBlock(
          (tdnn1): TDNNBlock(
            (conv): Conv1d(
              (conv): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
            )
            (activation): ReLU()
            (norm): BatchNorm1d(
              (norm): BatchNorm1d(1024, eps=1e-05, momentu

In [37]:

import torch
import torch.nn.functional as F 


def enroll_speaker(trials=2, trial_duration=5):
    pass_threshold = 0.45
    print(f"You will be asked to speak {trials} times, each time you can say whatever you want for {trial_duration} seconds, the model will save your voiceprint afterwards")
    embeddings = []
    for i in range(trials):
        input(f"Press Enter to Start trial {i+1}...")
        print(f"Trial number {i+1} : ")
        audio, sr = record_audio(16000, trial_duration)
        audio = torch.tensor(audio, dtype=torch.float32)
        audio = audio.T
        audio = audio.mean(dim=0, keepdim=True) # in case of multiple channels, ecapa accepts mono audio with sr = 16000
        audio = torchaudio.functional.resample(audio, sr, 16000)
        embeddings.append(classifier.encode_batch(audio, normalize=False))
        if i > 0:
            similarity = F.cosine_similarity(embeddings[-1][0,:,:], embeddings[-2][0,:,:])
            assert similarity.item() > pass_threshold, "Your recorded voice is very different from the previous, try again"
    res = torch.cat(embeddings, dim=0)
    res = F.normalize(res, p=2, dim=-1)
    print(f"Finished Enrollment Procedure")
    return F.normalize(res.mean(dim=0), dim=-1)



def authorized_speaker(audio, sr, ref_dim:torch.tensor) -> bool:
    """Checks the similarity with the enrolled speaker """
    if not isinstance(audio, torch.Tensor):
        audio = torch.tensor(audio, dtype=torch.float32)
        if audio.shape[0] != 1:
            audio = audio.T
    print(audio.shape)
    audio = audio.mean(dim=0, keepdim=True)
    audio = torchaudio.functional.resample(audio, sr, 16000)
    audio_dim = classifier.encode_batch(audio, normalize=False)[0]
    audio_dim = F.normalize(audio_dim, p=2, dim=-1)
    sim = F.cosine_similarity(audio_dim, ref_dim)
    print(f"{sim=}")
    return sim.item() > 0.45




In [38]:
authorized_spk_dim = enroll_speaker(trials=3, trial_duration=5)

You will be asked to speak 3 times, each time you can say whatever you want for 5 seconds, the model will save your voiceprint afterwards


Trial number 1 : 
Recording audio for 5 seconds...
Trial number 2 : 
Recording audio for 5 seconds...
Trial number 3 : 
Recording audio for 5 seconds...
Finished Enrollment Procedure


In [46]:
audio = record_audio(16000, 5)

Recording audio for 5 seconds...


In [47]:
authorized_speaker(audio[0], audio[1], authorized_spk_dim)

torch.Size([1, 80000])
sim=tensor([0.0292], device='cuda:0')


False

### The speaker verification is working fine for now (needs further testing to check the authorization).
### The next step is to add it to the pipeline : Audio -> Speaker authorization -> ASR -> LLM -> Response

In [25]:
audio = record_audio(16000, 10)
if not authorized_speaker(audio[0], audio[1], authorized_spk_dim):
    raise Exception('Unauthorized speaker.')
text = model.transcribe(audio=audio, language="English")[0].text
messages = [HumanMessage(content=text)]
tools = build_registry_tools(registry)
response = llm_chat(messages=messages, tools=tools)

Recording audio for 10 seconds...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


torch.Size([1, 160000])
sim=tensor([0.7155], device='cuda:0')


HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[AGENT]: Hello Shane! Welcome to Reddit Stories. It's great to have you here today. 

Dino Week is a fantastic theme - dinosaurs are such an exciting topic for storytelling! I'm ready to help you create content or scripts for your show. 

What would you like to work on today? Are you looking to:
- Create a new script about dinosaurs?
- Search for existing scripts related to dinosaurs?
- Or perhaps something else entirely?

Let me know how I can assist with your Reddit Stories episode!


In [54]:
import speech_recongition as sr

ModuleNotFoundError: No module named 'speech_recongition'